In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
!pip install -q pytorch-fid
import os
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from google.colab import drive
BATCH_SIZE = 128

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 64x64 -> 32x32, no BatchNorm on the first layer
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),

            # 32x32 -> 16x16
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # 16x16 -> 8x8
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            # 8x8 -> 4x4
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),

            # 4x4 -> 1x1, collapse to a single value
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0),
        )

    def forward(self, x):
        # x: (batch, 3, 64, 64) -> (batch,) real/fake probability
        return self.net(x).view(-1)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            # 1x1 latent -> 4x4, project noise up into a small grid
            nn.ConvTranspose2d(latent_dim, 512, kernel_size=4, stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            # 4x4 -> 8x8
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            # 8x8 -> 16x16
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 16x16 -> 32x32
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # 32x32 -> 64x64, output 3 channels
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        # z: (batch, latent_dim, 1, 1) -> (batch, 3, 64, 64)
        return self.net(z)

In [ ]:
NOISE_DIM = 100

generator = Generator(latent_dim=NOISE_DIM)
discriminator = Discriminator()

def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)
generator.apply(weights_init)
discriminator.apply(weights_init)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
generator = generator.to(device)
discriminator = discriminator.to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
LR = 2e-4
opt_d = optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.999))
opt_g = optim.Adam(generator.parameters(),     lr=LR, betas=(0.5, 0.999))

In [ ]:
drive.mount('/content/drive')
!ls "/content/drive/MyDrive/dog-project-data"
!ls "/content/drive/MyDrive/dog-project-data/processed"

!cp '/content/drive/MyDrive/dog-project-data/processed/dogs64.npy' '/content/'
df = np.load('/content/dogs64.npy')
csv_data = pd.read_csv('/content/drive/MyDrive/dog-project-data/processed/metadata.csv')

train = csv_data.index[csv_data["split"] == "train"].tolist()
val   = csv_data.index[csv_data["split"] == "val"].tolist()
test = csv_data.index[csv_data["split"] == "test"].tolist()

# getting the actual images from the indexs
train_images = df[train]
val_images = df[val]
test_images = df[test]

del df # free up memory

train_images = train_images.transpose(0, 3, 1, 2)
val_images = val_images.transpose(0, 3, 1, 2)
test_images = test_images.transpose(0, 3, 1, 2)
#print(val_images.shape)

train_images = np.ascontiguousarray(train_images)
val_images = np.ascontiguousarray(val_images)
test_images = np.ascontiguousarray(test_images)

# convert to tensor from numpy array
train_images = torch.from_numpy(train_images)
val_images = torch.from_numpy(val_images)
test_images = torch.from_numpy(test_images)

train_dataset = TensorDataset(train_images)
val_dataset = TensorDataset(val_images)
test_dataset = TensorDataset(test_images)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last = True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last = False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
from PIL import Image

def build_reference_folder(images_uint8, out_dir):
    """images_uint8: (N, 3, 64, 64) uint8 array (channels-first). Writes PNGs."""
    os.makedirs(out_dir, exist_ok=True)
    for i, img in enumerate(images_uint8):
        img = np.transpose(img, (1, 2, 0))        # CHW -> HWC for PIL
        Image.fromarray(img).save(f'{out_dir}/{i:05d}.png')

# build once; val_images etc. are the uint8 channels-first arrays from your data cell
VAL_REF_DIR = '/content/fid_ref/val'
build_reference_folder(val_images.numpy(), VAL_REF_DIR)

In [ ]:
import shutil
import torch
from pytorch_fid import fid_score

def compute_fid_for_epoch(generator, n_samples=5000, ref_dir=VAL_REF_DIR):
    gen_dir = '/content/fid_gen'
    shutil.rmtree(gen_dir, ignore_errors=True)   # clear last epoch's samples
    os.makedirs(gen_dir, exist_ok=True)

    generator.eval()
    written = 0
    with torch.no_grad():
        while written < n_samples:
            b = min(BATCH_SIZE, n_samples - written)
            z = torch.randn(b, NOISE_DIM, 1, 1, device=device)
            fake = generator(z)                          # [-1, 1] float, (b,3,64,64)
            fake = (fake + 1) * 127.5                     # invert normalization -> [0,255]
            fake = fake.clamp(0, 255).to(torch.uint8).cpu().numpy()
            for img in fake:
                img = np.transpose(img, (1, 2, 0))        # CHW -> HWC
                Image.fromarray(img).save(f'{gen_dir}/{written:05d}.png')
                written += 1

    fid = fid_score.calculate_fid_given_paths(
        [gen_dir, ref_dir],
        batch_size=50,
        device=device,
        dims=2048,
    )
    return fid

In [ ]:
CKPT_DIR = '/content/drive/MyDrive/dog-project-data/checkpoints/gan'
import os
os.makedirs(CKPT_DIR, exist_ok=True)

NUM_EPOCHS = 25
fid_history = []

# --- training ---
for epoch in range(NUM_EPOCHS):
    generator.train()
    discriminator.train()

    for (real_imgs,) in train_loader:
        real_imgs = real_imgs.to(device).float() / 127.5 - 1   # uint8 [0,255] -> [-1,1]
        b = real_imgs.size(0)

        # --- Step 1: discriminator ---
        opt_d.zero_grad()

        real_logits = discriminator(real_imgs)
        loss_real = criterion(real_logits, torch.ones(b, device=device))

        z = torch.randn(b, NOISE_DIM, 1, 1, device=device)
        fake_imgs = generator(z)
        fake_logits = discriminator(fake_imgs.detach())
        loss_fake = criterion(fake_logits, torch.zeros(b, device=device))

        loss_d = loss_real + loss_fake
        loss_d.backward()
        opt_d.step()

        # --- Step 2: generator ---
        opt_g.zero_grad()
        gen_logits = discriminator(fake_imgs)
        loss_g = criterion(gen_logits, torch.ones(b, device=device))
        loss_g.backward()
        opt_g.step()

   

    generator.eval()
    fid = compute_fid_for_epoch(generator)   # STUB: sample 5000, pytorch-fid vs val reference folder
    fid_history.append((epoch, fid))
    
     # --- per-epoch: checkpoint + FID vs val ---
    torch.save({'epoch': epoch, 'G': generator.state_dict(), 'D': discriminator.state_dict(),
            'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict(), 'fid_history': fid_history},
           f'{CKPT_DIR}/ckpt_epoch_{epoch:02d}.pt')

    print(f'epoch {epoch:02d}  loss_d {loss_d.item():.3f}  loss_g {loss_g.item():.3f}  FID {fid:.2f}')
    with open(f'{CKPT_DIR}/train_log.txt', 'a') as f:
        f.write(f'epoch {epoch:02d}  loss_d {loss_d.item():.3f}  loss_g {loss_g.item():.3f}  FID {fid:.2f}\n')

# --- FID vs epoch graph ---
epochs, fids = zip(*fid_history)
plt.figure(figsize=(7, 4))
plt.plot(epochs, fids, marker='o')
plt.xlabel('epoch')
plt.ylabel('FID (vs val)')
plt.title('FID vs epoch, GAN')
plt.grid(True, alpha=0.3)
plt.savefig(f'{CKPT_DIR}/fid_curve.png', dpi=120, bbox_inches='tight')
plt.show()

# save raw FID values so the curve can share an axis with Alfonso's VAE curve
import csv
with open(f'{CKPT_DIR}/fid_history.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['epoch', 'fid'])
    w.writerows(fid_history)

In [ ]:
# --- Final eval: 10k samples, FID vs train+val and vs test, save to Drive ---
def build_reference_folder(images_uint8, out_dir, prefix=''):
    os.makedirs(out_dir, exist_ok=True)
    for i, img in enumerate(images_uint8):
        Image.fromarray(np.transpose(img, (1, 2, 0))).save(f'{out_dir}/{prefix}{i:05d}.png')

TRAINVAL_REF_DIR = '/content/fid_ref/train_val'
TEST_REF_DIR = '/content/fid_ref/test'
build_reference_folder(train_images.numpy(), TRAINVAL_REF_DIR, prefix='train_')
build_reference_folder(val_images.numpy(),   TRAINVAL_REF_DIR, prefix='val_')
build_reference_folder(test_images.numpy(),  TEST_REF_DIR)

# generate 10k once, use the same samples for both FIDs
FINAL_GEN_DIR = '/content/final_gen_gan'
shutil.rmtree(FINAL_GEN_DIR, ignore_errors=True)
os.makedirs(FINAL_GEN_DIR)
generator.eval()
written = 0
with torch.no_grad():
    while written < 10_000:
        b = min(BATCH_SIZE, 10_000 - written)
        z = torch.randn(b, NOISE_DIM, 1, 1, device=device)
        fake = ((generator(z) + 1) * 127.5).clamp(0, 255).to(torch.uint8)
        for img in fake.permute(0, 2, 3, 1).cpu().numpy():
            Image.fromarray(img).save(f'{FINAL_GEN_DIR}/{written:05d}.png')
            written += 1

fid_trainval = fid_score.calculate_fid_given_paths([FINAL_GEN_DIR, TRAINVAL_REF_DIR], batch_size=50, device=device, dims=2048)
fid_test     = fid_score.calculate_fid_given_paths([FINAL_GEN_DIR, TEST_REF_DIR],     batch_size=50, device=device, dims=2048)
print(f'Final FID  train+val: {fid_trainval:.2f}   test: {fid_test:.2f}')

with open(f'{CKPT_DIR}/final_fid.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['split', 'fid'])
    w.writerows([('train+val', fid_trainval), ('test', fid_test)])

# step 5: copy the 10k samples to Drive (fix the path to match your shared folder)
SAMPLES_DRIVE_DIR = '/content/drive/MyDrive/dog-project-data/midterm generated samples/gan'
shutil.copytree(FINAL_GEN_DIR, SAMPLES_DRIVE_DIR, dirs_exist_ok=True)